In [2]:
# Load new data
df_new = spark.read.csv("Files/bronze/pandas_df.csv", header=True, inferSchema=True)

# Remove duplicates based on the key (Order_ID or your unique key)
df_new_dedup = df_new.dropDuplicates(["Order_ID"])

# Write deduplicated staging table
df_new_dedup.write.format("delta").mode("overwrite").saveAsTable("staging_sales")

# Now run the MERGE
from delta.tables import DeltaTable

delta_main = DeltaTable.forName(spark, "sales_transformed")

delta_main.alias("main").merge(
    spark.table("staging_sales").alias("new"),
    "main.Order_ID = new.Order_ID"  # Your key
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("CDC MERGE applied successfully after deduplication!")

StatementMeta(, adb804e4-0856-4ee3-b3af-6dd6224be731, 4, Finished, Available, Finished)

CDC MERGE applied successfully after deduplication!
